<a href="https://colab.research.google.com/github/seun829/DataScience/blob/main/Unit7/Copy_of_Unit7ExercisesSF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fitting Curves: Concepts

What you'll do:

- Answer questions about what a GP is, and its relationship to GLMs and splines.
- Practice applying each of: polynomial modeling, b splines, and GPs
- You'll get a chance to read about and try to comprehend a more standard implementation of a GP.

Have fun!

In [ ]:
!pip install preliz
!pip install bambi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.2/529.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.6 MB/s eta 0:00:00
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully uninstalled llvmlite-0.43.0
  Attempting uninstall: numba
    Found existing installation: numba 0.60.0
    Uninstalling numba-0.60.0:
      Successfully uninstalled numba-0.60.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.6/109.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.9/218.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.4/259.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
import arviz as az
import xarray as xr
import pymc as pm
from scipy.interpolate import PchipInterpolator

In [ ]:
import preliz as pz

In [ ]:
import bambi as bmb

**Task1**:

Why would you ever want to include a polynomial element in a model you built? What's the benefit of using polynomials to model?

Having polynomial elements in a model allows for more nuanced correlations that can only be modeled by non linear based models.



**Task2**:

Why would you ever NOT want to include a polynomial element in a model you built?

When the correlation is a lot more obvious and defined as a straight line or close to it. In this case, having a model made for curves would not be optimal.

**Task3**:

What's the point of using b splines?

Using b splines allows us to find the flexibility of our polynomial regression. It involves splitting the x axis up into pieces, and then using polynomial regression for each part.

**Task4**:

Describe what a Gaussian Process is, in your own words. *Don't worry about being correct, just try to explain it to yourself*. I will not grade this question for accuracy.

A Gaussian Process is used to measure the true relationship between all of the different points. You have to choose a kernal method and check many different combinations. It basically is finding similarity between points and the extent to which this happens.

**Task5**:

Fit three models to the howell data (from Unit5ExercisesSF): polynomial, b splines, and Gaussian Process.

Plot the posterior predictive check on a scatter plot, as is standard/required.

Hint: Distributional models (variable variance) work better on the howell data.


In [ ]:
#downloads the data from my github
howell = pd.read_csv('https://raw.githubusercontent.com/thedarredondo/data-science-fundamentals/main/Data/howell.csv')
howell

,height,weight,age,male
0,151.765,47.825606,63.0,1
1,139.700,36.485807,63.0,0
2,136.525,31.864838,65.0,0
3,156.845,53.041914,41.0,1
4,145.415,41.276872,51.0,0
...,...,...,...,...
539,145.415,31.127751,17.0,1
540,162.560,52.163080,31.0,1
541,156.210,54.062497,21.0,0
542,71.120,8.051258,0.0,1


In [ ]:
how_model = bmb.Model("weight ~ poly(height, degree=7)", howell, family="negativebinomial")
idata_how_model = how_model.fit()

In [ ]:
bmb.interpret.plot_predictions(how_model, idata_how_model, "weight", pps=True)

plt.plot(howell.height, howell.weight, "C2.", zorder=-3)

In [ ]:
how_model2 = bmb.Model("weight ~ bs(height, degree=7, knots=knots)", howell, family="negativebinomial")
idata_how_model2 = how_model.fit()

In [ ]:
bmb.interpret.plot_predictions(how_model2, idata_how_model2, "weight", pps=True)

plt.plot(howell.height, howell.weight, ".C2", zorder=-3)

In [ ]:
#ig stands for inverse gamma
def get_ig_params(x_vals, l_b=None, u_b=None, mass=0.96, plot=False):
    """
    Returns a weakly informative prior for the length-scale parameter of the GP kernel.
    """

    differences = np.abs(np.subtract.outer(x_vals, x_vals))
    if l_b is None:
        l_b = np.min(differences[differences != 0]) * 2
    if u_b is None:
        u_b = np.max(differences) / 1.5

    dist = pz.InverseGamma()
    pz.maxent(dist, l_b, u_b, mass, plot=plot)

    return dict(zip(dist.param_names, dist.params))

In [ ]:
with pm.Model() as howell_model3:
  ℓ = pm.InverseGamma('ℓ', **get_ig_params(howell.height))

  #this is our kernal, which decides how our points relate to one another
  cov = pm.gp.cov.ExpQuad(1, ls=ℓ)
  #this specfices that we're using an HSGP
  gp = pm.gp.HSGP(m=[168], c=8.0, cov_func=cov)

  #f is for function, as in the function we use to transform our data
  f = gp.prior('f', X=howell.height)
  #prior for the neg binomial
  α = pm.HalfNormal('α', 1)
  #likelihood
  y = pm.NegativeBinomial("y", np.exp(f), α, observed=howell.weight)

  idata_how_model3 = pm.sample()


In [ ]:
_, ax = bmb.interpret.plot_predictions(howell_model3, idata_how_model3, ["hour"],
                                       pps = True,
                                       fig_kwargs={"figsize": (10, 3)})
ax[0].plot(howell["height"].values, howell["weight"].values, "C2.")

**Task6**:

Read the article on the pymc website about GP implementation on the Mauna Loa CO$_{2}$ data combined with CO$_{2}$ ice core data from the south pole.
[Link here.](https://www.pymc.io/projects/examples/en/latest/gaussian_processes/GP-MaunaLoa2.html)

Write down one thing you learned about GPs from reading the article.

Note: You probably won't understand much in this article--I had to read it about five times before I figured out what was going on. The points of this task are to: hammer home that good GP implementations are extremely technical, and increasure your exposure to the kinds of problems traditional GPs are good at solving.

So as expected I wasn't able to understand a lot from this article, but I did learn that the piecewise linear function is a type of GP function that can adapt to sudden trends in the data, like if the temperature in Austin one day suddenly shot up. This is done by changing the slope and intercept at certain "changepoints"

**Task7**:

Describe your favorite graph from the article in the previous task with as much technical detail as you can muster.

Explain why its your favorite.

My favorite graph was the one using the mean function to have certian changepoints to better model the data. It did choose where the changepoints were, which is not ideal considering the changepoints should be modeled by a behavior in the form of a covariance function. As a result, it's a smoother curve but not as realistic in a real world context. If it did use a covariance function, it would apply the formula of k(x, x) to scale based on known and unknown parameters.